# Ngrams

N-grams, according to Eder, are inferior to most-frequent word (MWF) analysis of a text, but can be effective with noisy data. The focus of Eder on English means that one other advantage, the efficacy in more inflected languages, shouldn't be overlooked.

First, to retrieve the data. We don't need the pre-tokenized data, since it's not to be parsed.

In [27]:
import pandas as pd
import random
import pprint
import logging
import numpy

# Note that the naming conventions need to be fixed: this includes all texts!
base_data = pd.read_csv("caesar_text_perseus.csv", sep="|")

print(base_data.info())
print("-" * 40)
print(base_data.iloc[1,])

<class 'pandas.DataFrame'>
RangeIndex: 2151 entries, 0 to 2150
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  2151 non-null   int64
 1   work        2151 non-null   str  
 2   book        2151 non-null   int64
 3   text        2151 non-null   str  
 4   src         2151 non-null   str  
 5   chapter     2151 non-null   int64
dtypes: int64(3), str(3)
memory usage: 101.0 KB
None
----------------------------------------
Unnamed: 0                                                    1
work                                                     gallic
book                                                          1
text          Apud Helvetios longe nobilissimus fuit et diti...
src            urn:cts:latinLit:phi0448.phi001.perseus-lat2:1.2
chapter                                                       1
Name: 1, dtype: object


I need to make sure there aren't subtly different characters due to encoding differences.

In [2]:
def unique_ch(input_str):
    collector = []
    for char in input_str:
        if char in collector:
            continue
        collector.append(char)
    return collector

all_text = " ".join(base_data["text"].to_list())
print(sorted(unique_ch(all_text)))

[' ', '!', '"', "'", '(', ')', ',', '-', '.', '/', '0', '2', '3', '5', '6', '7', '8', ':', ';', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'X', 'Z', '[', '\\', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '·', '̅', 'Ι', '—', '‘', '’', '“', '”', '†']


Nothing too special to see there.

Now we need to create a function that allows us to divide up the text in two ways.

One is to do by citation, the other is to further subdivide into equal groups.

In [3]:
import re

# Remove non-alphabetic characters
def clean(input_str):
    val = re.sub(pattern="[^a-zA-Z ]", repl="", string=input_str).casefold()
    return re.sub(" +", " ", val)

print(clean(base_data["text"][0]))

gallia est omnis divisa in partes tres quarum unam incolunt belgae aliam aquitani tertiam qui ipsorum lingua celtae nostra galli appellantur hi omnes lingua institutis legibus inter se differunt gallos ab aquitanis garumna flumen a belgis matrona et sequana dividit horum omnium fortissimi sunt belgae propterea quod a cultu atque humanitate provinciae longissime absunt minimeque ad eos mercatores saepe commeant atque ea quae ad effeminandos animos pertinent important proximique sunt germanis qui trans rhenum incolunt quibuscum continenter bellum gerunt qua de causa helvetii quoque reliquos gallos virtute praecedunt quod fere cotidianis proeliis cum germanis contendunt cum aut suis finibus eos prohibent aut ipsi in eorum finibus bellum gerunt eorum una pars quam gallos obtinere dictum est initium capit a flumine rhodano continetur garumna flumine oceano finibus belgarum attingit etiam ab sequanis et helvetiis flumen rhenum vergit ad septentriones belgae ab extremis galliae finibus oriunt

In [4]:
# get ngrams, keeping word boundaries as a character
def ngram(text, n) -> list[str]:
        return [text[i:(i + n)] for i in range(len(text)) if (i + n - 1) < len(text)]

print(ngram("Gallia est omnis divisa in partes tres", 2))
print(ngram("Gallia est omnis divisa in partes tres", 3))

['Ga', 'al', 'll', 'li', 'ia', 'a ', ' e', 'es', 'st', 't ', ' o', 'om', 'mn', 'ni', 'is', 's ', ' d', 'di', 'iv', 'vi', 'is', 'sa', 'a ', ' i', 'in', 'n ', ' p', 'pa', 'ar', 'rt', 'te', 'es', 's ', ' t', 'tr', 're', 'es']
['Gal', 'all', 'lli', 'lia', 'ia ', 'a e', ' es', 'est', 'st ', 't o', ' om', 'omn', 'mni', 'nis', 'is ', 's d', ' di', 'div', 'ivi', 'vis', 'isa', 'sa ', 'a i', ' in', 'in ', 'n p', ' pa', 'par', 'art', 'rte', 'tes', 'es ', 's t', ' tr', 'tre', 'res']


In [5]:
def tokenize(text: str):
    """
    Ensures the text is clean before tokenizing
    Will make debugging easier when I can guarantee
    tokenization is done the same way every time"""
    return clean(text).split(" ")

In [65]:
# Utility for next cell

def group_str(text: str, size: int):
    tokenized = tokenize(text)
    n = len(tokenized) // size
    assert n > 0, "size variable is larger than number of words in the provided text segment"

    indices = list(range(0, size * (n + 1), size))
    if len(tokenized) % size != 0:
        indices.append(len(tokenized))

    return [" ".join(tokenized[indices[i]:indices[i + 1]]) for i in range(0, len(indices) - 1)]

test = clean(base_data['text'][0])
grouped_test = group_str(text = test, size = 20)
print(grouped_test)

print(group_str(text = test, size = 178)) # Still works!

# This should throw an error
try:
    print(group_str(text = test, size = 179))
except AssertionError:
    print("Function failed successfully!")
else:
    AssertionError("Function should have failed, but did not.")

['gallia est omnis divisa in partes tres quarum unam incolunt belgae aliam aquitani tertiam qui ipsorum lingua celtae nostra galli', 'appellantur hi omnes lingua institutis legibus inter se differunt gallos ab aquitanis garumna flumen a belgis matrona et sequana dividit', 'horum omnium fortissimi sunt belgae propterea quod a cultu atque humanitate provinciae longissime absunt minimeque ad eos mercatores saepe commeant', 'atque ea quae ad effeminandos animos pertinent important proximique sunt germanis qui trans rhenum incolunt quibuscum continenter bellum gerunt qua', 'de causa helvetii quoque reliquos gallos virtute praecedunt quod fere cotidianis proeliis cum germanis contendunt cum aut suis finibus eos', 'prohibent aut ipsi in eorum finibus bellum gerunt eorum una pars quam gallos obtinere dictum est initium capit a flumine', 'rhodano continetur garumna flumine oceano finibus belgarum attingit etiam ab sequanis et helvetiis flumen rhenum vergit ad septentriones belgae ab', 'extremis

In [7]:
# Function that returns a list of data structures of two components: a string, and a citation.

class Segment:
    def __init__(self, cite, string):
        self.cite = cite
        self.string = string

def get_groups(df, cite: str="book", size: int=0) -> Segment:
    """
    Return groups of cleaned text by book or section, which is then further divided into texts of equal sizes if desired.

    @param df: pandas DataFrame csv
    @param cite: 'book' or 'work'
    @param size: Default of 0 keeps each book/work intact. An integer gives the maximum size each subsection should be, if dividing the text up is desired"""
    
    # Concatenate rows together by grouping (whether book or work)
    # During this loop, add to a Segment with the citation in a list
    grouping = []
    if cite == "book":
        grouping = ['work', cite]
    elif cite == "chapter":
        grouping = ['work', 'book', cite]
    else:
        AssertionError("Must use 'work' or 'chapter' for cite")
    grouped = df.groupby(grouping)

    citation_segments = [Segment(":".join([str(x) for x in name]), clean(" ".join(group['text'].to_list()))) for name, group in grouped]
        
    # format of subsection is [index]s[size] at the end of the citation, separated from the rest of the citation by a colon
    if size > 0:
        final_segments = []
        for s in citation_segments:
            final_segments += [Segment(s.cite + ":" + str(i) + "*" + str(size), x) for i, x in enumerate(group_str(s.string, size))]
        return final_segments
    else:
        return citation_segments


segments = get_groups(base_data, "book")
print(segments[0].cite, "\n", "-" * 100, '\n', segments[0].string[0:400])
print("-" * 100)
output = get_groups(base_data, "chapter", 10)
print(output[1].cite, "\n", output[1].string)
       


african:1 
 ---------------------------------------------------------------------------------------------------- 
 caesar itineribus iustis confectis nullo die intermisso a d xiiii kal ian lilybaeum pervenit statimque ostendit sese navis velle conscendere cum non amplius legionem tironum haberet unam equites vix dc tabernaculum secundum litus ipsum constituit ut prope fluctus verberaret hoc eo consilio fecit ne quis sibi morae quicquam fore speraret et ut omnes in dies horasque parati essent incidit per id tem
----------------------------------------------------------------------------------------------------
african:1:0:1*10 
 kal ian lilybaeum pervenit statimque ostendit sese navis velle conscendere


We need a better idea of what the final dataset will look like. Each column will be an n-gram, and each row will be a text. We need a list of all unique n-grams first. This should be put into a function once my initial exploration is done.

In [8]:
all_texts = clean(" ".join(base_data["text"].to_list()))

print(all_texts[1000:1400])   
print("Length: ", len(all_texts))

ur pertinent ad inferiorem partem fluminis rheni spectant in septentrionem et orientem solem aquitania a garumna flumine ad pyrenaeos montes et eam partem oceani quae est ad hispaniam pertinet spectat inter occasum solis et septentriones apud helvetios longe nobilissimus fuit et ditissimus orgetorix is m messala et p m pisone consulibus regni cupiditate inductus coniurationem nobilitatis fecit et 
Length:  1727788


In [9]:
all_ngrams = ngram(all_texts, 2)

print("Number of total 2grams: ", len(all_ngrams))

unique_ngrams = set(all_ngrams)

print("Number of unique 2grams: ", len(unique_ngrams))
print("-" * 120)
print(sorted(unique_ngrams))

Number of total 2grams:  1727787
Number of unique 2grams:  432
------------------------------------------------------------------------------------------------------------------------
[' a', ' b', ' c', ' d', ' e', ' f', ' g', ' h', ' i', ' k', ' l', ' m', ' n', ' o', ' p', ' q', ' r', ' s', ' t', ' u', ' v', ' x', ' z', 'a ', 'aa', 'ab', 'ac', 'ad', 'ae', 'af', 'ag', 'ah', 'ai', 'al', 'am', 'an', 'ao', 'ap', 'aq', 'ar', 'as', 'at', 'au', 'av', 'ax', 'az', 'b ', 'ba', 'bd', 'be', 'bf', 'bh', 'bi', 'bl', 'bm', 'bn', 'bo', 'bp', 'br', 'bs', 'bt', 'bu', 'bv', 'by', 'c ', 'ca', 'cc', 'ce', 'cf', 'ch', 'ci', 'cl', 'cm', 'cn', 'co', 'cp', 'cq', 'cr', 'cs', 'ct', 'cu', 'cv', 'cx', 'cy', 'd ', 'da', 'db', 'dc', 'dd', 'de', 'df', 'dg', 'dh', 'di', 'dl', 'dm', 'dn', 'do', 'dp', 'dq', 'dr', 'ds', 'dt', 'du', 'dv', 'dy', 'e ', 'ea', 'eb', 'ec', 'ed', 'ee', 'ef', 'eg', 'eh', 'ei', 'el', 'em', 'en', 'eo', 'ep', 'eq', 'er', 'es', 'et', 'eu', 'ev', 'ex', 'f ', 'fa', 'fe', 'ff', 'fi', 'fl', 'fo', 'fr',

In [10]:
def single_ngram(ngram: str, raw_text: str):
    """
    Debugging function for verifying the raw frequency of an ngram in a set of text"""
    ngrams = ngram(raw_text)
    

In [11]:
# How to count how many times each item in the set above appears in a text segment?

# TODO WHAT HAPPENS IF WE WANT TO USE A GROUPED SET OF TEXTS?

# First, a basic function
# Note the following issue: are we going to want to use multiple values for n?
def count_ngrams(set_of_all: set, text: str, n: int=2, citation="", norm: bool=True) -> dict:
    """
    Given a "dictionary" with all the possible ngrams in @param set_of_all, 
    how many of each ngram occurs in the given raw text? Must provide n for
    number of ngrams at the moment, which has to match the ngram size in
    @param set_of_all. @param norm lets the user decide whether or not to
    normalize, which the function does by default.

    TODO NEEDSDOC
    @param set_of_all: A set of all the possible n-grams in all texts
    @param text: a text from one text segment targeted for ngram analysis"""

    # some assertions to avoid nasty issues later
    debug_length: int = len(list(set_of_all)[0])
    for item in list(set_of_all):
        if len(item) != debug_length:
            raise AssertionError("set_of_all argument must provide only ngrams of the same length")
    assert len(list(set_of_all)[0]) == n, "ngrams provided are not equal to n"
    
    # clean just in case, since it should be an idempotent function
    text = clean(text)

    ngrams = ngram(text, n)
    
    # For normalization
    length = len(tokenize(text))

    # save normalized frequencies into dictionary, or into a data frame? Dictionary for now
    results = {}
    if len(citation) > 0:
        results["work"] = citation
    for ng in list(set_of_all):
        norm_factor: int = 1
        if norm:
            norm_factor: float = 1000 / length
        results[ng] = ngrams.count(ng) * norm_factor

    return results
random.seed(a="caesarisfun")
rand_seg = random.choice(segments)
logging.debug(pprint.pp(count_ngrams(unique_ngrams, rand_seg.string, citation=rand_seg.cite, norm=False), width = 200, compact=True, sort_dicts=True))



{' a': 366,
 ' b': 40,
 ' c': 426,
 ' d': 129,
 ' e': 343,
 ' f': 127,
 ' g': 47,
 ' h': 113,
 ' i': 288,
 ' k': 0,
 ' l': 95,
 ' m': 153,
 ' n': 191,
 ' o': 115,
 ' p': 307,
 ' q': 163,
 ' r': 127,
 ' s': 245,
 ' t': 107,
 ' u': 75,
 ' v': 125,
 ' x': 4,
 ' z': 0,
 'a ': 227,
 'aa': 0,
 'ab': 80,
 'ac': 115,
 'ad': 97,
 'ae': 155,
 'af': 0,
 'ag': 39,
 'ah': 0,
 'ai': 3,
 'al': 87,
 'am': 122,
 'an': 186,
 'ao': 0,
 'ap': 18,
 'aq': 23,
 'ar': 161,
 'as': 117,
 'at': 311,
 'au': 69,
 'av': 49,
 'ax': 7,
 'az': 0,
 'b ': 24,
 'ba': 89,
 'bd': 0,
 'be': 59,
 'bf': 0,
 'bh': 0,
 'bi': 58,
 'bl': 2,
 'bm': 1,
 'bn': 0,
 'bo': 6,
 'bp': 0,
 'br': 9,
 'bs': 17,
 'bt': 0,
 'bu': 145,
 'bv': 0,
 'by': 0,
 'c ': 56,
 'ca': 111,
 'cc': 28,
 'ce': 97,
 'cf': 0,
 'ch': 0,
 'ci': 163,
 'cl': 15,
 'cm': 0,
 'cn': 0,
 'co': 235,
 'cp': 0,
 'cq': 0,
 'cr': 24,
 'cs': 0,
 'ct': 101,
 'cu': 109,
 'cv': 0,
 'cx': 1,
 'cy': 0,
 'd ': 113,
 'da': 36,
 'db': 0,
 'dc': 1,
 'dd': 3,
 'de': 143,
 'df': 2,
 'd

In [52]:
# Okay, how to put all this together? It looks like pandas supports creating data frames from a list of dicts:

dicts = [count_ngrams(unique_ngrams, s.string, n=2, citation=s.cite, norm=True) for s in segments]

results = pd.DataFrame(dicts)
results

,work,hi,ho,hm,hl,ha,he,hy,hs,hr,...,fl,fo,fr,fu,vr,f,kw,ka,ko,vb
0,african:1,5.389175,12.395104,0.000000,0.000000,10.855339,0.307953,0.384941,0.384941,0.076988,...,0.538918,2.848564,6.159058,8.391716,0.000000,0.000000,0.000000,0.461929,0.000000,0.000000
1,alexandrine:1,6.054200,12.012301,0.000000,0.000000,11.531809,1.345378,0.288295,0.096098,1.633673,...,4.997117,4.516625,2.402460,5.958101,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,amicitia:1,9.571951,11.400301,0.000000,0.000000,10.432351,1.398150,0.107550,0.000000,0.215100,...,1.398150,2.581200,2.903850,6.883201,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,brutus:1,12.494994,10.893072,0.120144,0.000000,9.251101,4.725671,0.520625,0.000000,0.280336,...,2.683220,3.043652,2.042451,12.695234,0.040048,0.640769,0.040048,0.000000,0.040048,0.040048
4,catilinae_sallusti:1,6.469761,11.533052,0.000000,0.000000,11.064229,3.094233,0.000000,0.000000,0.187529,...,1.875293,7.594937,2.062822,7.688701,0.000000,0.000000,0.000000,0.281294,0.000000,0.000000
5,civil:1,10.289389,12.861736,0.000000,0.000000,9.646302,1.561782,0.000000,0.091870,0.000000,...,4.225999,2.388608,8.268259,5.328434,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,civil:2,7.674236,12.529366,0.000000,0.000000,10.493344,1.096319,0.000000,0.156617,0.000000,...,1.879405,3.288958,3.758810,8.144088,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,civil:3,12.445923,11.247920,0.066556,0.000000,11.114809,3.194676,0.066556,0.000000,0.199667,...,2.262895,3.394343,3.727121,6.988353,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,deiotaro:1,7.443532,14.117043,0.000000,0.000000,8.470226,1.540041,0.000000,0.000000,0.000000,...,0.770021,4.106776,3.336756,10.780287,0.000000,0.000000,0.000000,0.000000,0.000000,0.256674
9,gallic:1,8.076358,12.604014,0.000000,0.000000,14.561919,11.013216,0.000000,0.000000,0.000000,...,4.282917,1.713167,4.650024,5.261870,0.000000,0.000000,0.000000,0.122369,0.000000,0.000000


In [58]:
# Let's verify a random cell here

def verify_random_cell(df_results, df_data, n: int=2):
    """
    Must be an input in the format output from the previous cell
    DOESN'T WORK WITH NORMALIZING!!!!!"""
    assert df_results.iloc[:,1].dtypes.type is numpy.int64, "Data must not be normalized for this function"
    
    colname = str(random.choice(results.columns[1:]))

    title = df_results['work'].sample().to_list()[0]
 
    title_s = title.split(":")
    select = df_data
    if len(title_s) == 2:
        mask = (df_data['work'] == title_s[0]) & (df_data['book'] == int(title_s[1]))
        select = df_data.loc[mask]
    elif len(title_s) == 3:
        mask = (df_data['work'] == title_s[0]) & (df['book'] == int(title_s[1])) & (True if 's' in title else df_data['chapter'] == int(title_s[2]))
        select = df_data.loc[mask]
    elif len(title_s) == 4:
        mask = (df_data['work'] == title_s[0]) & (df_data['book'] == int(title_s[1])) & (df_data['chapter'] == int(title_s[2]))
        select = df_data.loc[mask]
    else:
        AssertionError(f"Work {title_s} is formatted incorrectly")

    size = 0
    index = 0
    if '*' in title:
        size_char = title_s[-1]
        size_char = size_char.split("s")
        size = int(size_char[1])
        index = int(size_char[0])

    logging.debug("Report:\n" + "*" * 40, "\n", f"Size: {size}\nIndex: {index}", '\n', "*" * 40)

    text = clean(" ".join(select["text"].to_list()))

    if size > 0:
        text = group_str(text, size)[index]

    ng = ngram(text, n)
    recount = ng.count(colname)
    logging.debug(colname)
    og_count = df_results.loc[df_results['work'] == title,colname].to_list()[0]
    logging.debug("Results:\n", "*" * 40, f"\nRecount: {recount}\nRetrieved from results: {og_count}")
    return [recount, og_count]


verify_random_cell(results, base_data)

AssertionError: Data must not be normalized for this function

In [40]:

# test many times that we get the same results with verify_... as we do with the normal function
for i in range(0,10000):
    test = verify_random_cell(results, base_data)
    assert test[0] == test[1], f"Mismatch where original count equals {test[1]} at iteration {i}"

Here is a reminder of the functions/code segments to unit test:
* ngram()
* conversion from dictionaries to data frame
* get_groups()
* tokenize()

We cannot make any sure conclusions about this yet, but I do want to get some data into R! So, let's get this output to a file.

In [68]:
results.to_csv("all_texts_ngrams_normalized.csv", index=False)

new_df = pd.read_csv("all_texts_ngrams_normalized.csv")

print(new_df.compare(results))

Empty DataFrame
Columns: []
Index: []
